# RSICCformer: Colab training and evaluation

This notebook clones the code from GitHub, prepares LEVIR-CC, and runs training, evaluation, and example caption generation. Code and data stay under `/content/RSICC`. Each run gets a separate Google Drive folder for checkpoints, settings, environment information, and results.

Select a GPU runtime in Colab. An earlier version completed training on an A100; the workflow changes in this notebook have not yet been tested in Colab. Adjust batch size and data-loading workers for your hardware.

Run the cells in order. To continue an earlier training run, set `RESUME_CHECKPOINT` in the configuration cell. Only load checkpoints you trust: RSICC checkpoints contain serialized Python objects.


## Configuration

Edit this cell before running the workflow. Use a different `DATASET_LABEL` for each dataset variant. Prepared data means RSICC HDF5/JSON training files; modified images with captions still use the raw-data mode. Existing Drive backups are used only when explicitly selected. Each new run gets a unique output folder.


In [ ]:
import os
import subprocess
from pathlib import Path

# Repository and output location (Drive output must be under MyDrive).
REPO_URL = "https://github.com/yakupgulcan/RSICC.git"
REPO_REF = "experiment/colab-workflow"  # Push the updated code before using this ref.
PROJECT_DIR = Path("/content/RSICC")
OUTPUT_ROOT = Path("/content/drive/MyDrive/RSICC_Outputs")
DATASET_LABEL = "levir-cc"  # Change this for each image/caption variant; no slashes.

# Choose the data mode explicitly.
USE_PREPROCESSED_DATA = False  # True: load ready HDF5/JSON; False: build from images/captions.
RAW_DATASET_SOURCE = "https://drive.google.com/file/d/1YgWES-0OOL-3KK4yIV3-y8MN-_c1ELEE/view?usp=drive_link"
LOCAL_DATASET_DIR = None  # Optional existing folder with LevirCCcaptions.json and images/.
PREPROCESSED_SOURCE = "/content/drive/MyDrive/RSICC/data_preprocessed.zip"  # Used only when True.
SAVE_PREPROCESSED_BACKUP = False  # Optional backup after building training files.
PREPROCESSED_BACKUP_PATH = None  # None: save in this run folder; existing files are never replaced.

# Settings for building HDF5/JSON. DATA_NAME must match an existing prepared archive.
CAPTIONS_PER_IMAGE = 5
MIN_WORD_FREQ = 5
MAX_LEN = 50
DATA_NAME = f"LEVIR_CC_{CAPTIONS_PER_IMAGE}_cap_per_img_{MIN_WORD_FREQ}_min_word_freq"

# Training and continuation.
RESUME_CHECKPOINT = None  # Full checkpoint path; keep compatible data and vocabulary when resuming.
TOTAL_EPOCHS = 20  # Total target, including epochs already completed.
BATCH_SIZE = 32
NUM_WORKERS = 4
STOP_CRITERIA = 10
PRINT_FREQ = 100  # Print first/final batch and every N batches.
ENCODER_LR = 1e-4
DECODER_LR = 1e-4

# Test evaluation and example caption generation.
EVAL_CHECKPOINT = None  # None: best checkpoint from this run; otherwise an existing checkpoint path.
EVAL_BEAM_SIZE = 1
EXAMPLE_IMAGE_A = PROJECT_DIR / "Example/A/train_000016.png"
EXAMPLE_IMAGE_B = PROJECT_DIR / "Example/B/train_000016.png"

if not DATASET_LABEL or any(c in DATASET_LABEL for c in "/\\") or DATASET_LABEL in (".", ".."):
    raise ValueError("DATASET_LABEL must be a nonempty folder label without slashes.")
if PRINT_FREQ < 1:
    raise ValueError("PRINT_FREQ must be positive.")
if not USE_PREPROCESSED_DATA and DATA_NAME != f"LEVIR_CC_{CAPTIONS_PER_IMAGE}_cap_per_img_{MIN_WORD_FREQ}_min_word_freq":
    raise ValueError("DATA_NAME must match the configured preprocessing settings.")


## 1. Check GPU and Java

Java is required for the METEOR metric.

In [ ]:
import torch
import subprocess

subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["java", "-version"], check=True)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab.")
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone the repository

Use the configuration cell to set `REPO_URL` and `REPO_REF` to the fork and branch, tag, or commit you want to run. The default branch below must be pushed to GitHub before Colab can use it. After merging these changes, you can use `main`; use a commit hash to reproduce a specific code version.

Start a new runtime if `/content/RSICC` already exists. After a runtime restart, restore the imports and configuration variables from these cells before continuing.


In [ ]:
if PROJECT_DIR.exists():
    raise FileExistsError(f"{PROJECT_DIR} already exists. Use a fresh runtime.")
subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=PROJECT_DIR, check=True)
os.chdir(PROJECT_DIR)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)
if not (PROJECT_DIR / "notebook_runtime.py").is_file():
    raise FileNotFoundError("Push the updated workflow and select that Git revision before running this notebook.")


## 4. Install dependencies and create a run folder

Colab software versions can change between sessions. If installation requests a runtime restart, restart and restore the configuration and imports before continuing.

Each execution of the run-folder cell creates a unique folder. Run it once per experiment. Data files are isolated per run; no old Drive backup is selected automatically. Your own image preprocessing is supported as long as the uploaded dataset keeps the expected LEVIR-CC caption schema and image layout.


In [ ]:
%pip install -r requirements.txt --quiet
%pip install gdown google-api-python-client google-auth --quiet

In [ ]:
import json
import os
import platform
import subprocess
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4
import torch
import torchvision

from notebook_runtime import run_logged, required_data_files, restore_preprocessed_data
import shutil
import sys
import zipfile

os.chdir(PROJECT_DIR)
RUN_ID = DATASET_LABEL + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
CHECKPOINT_DIR = PROJECT_DIR / "runs" / RUN_ID / "models_checkpoint"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=False)
DATA_DIR = PROJECT_DIR / "runs" / RUN_ID / "data"
DATA_DIR.mkdir()
DATASET_DIR = (Path(LOCAL_DATASET_DIR) if LOCAL_DATASET_DIR is not None
               else PROJECT_DIR / "runs" / RUN_ID / "raw_dataset")
DRIVE_DATA_BACKUP = (Path(PREPROCESSED_BACKUP_PATH) if PREPROCESSED_BACKUP_PATH is not None
                     else RUN_DIR / "data_preprocessed.zip")
notebook_config = {
    "repo_url": REPO_URL, "repo_ref": REPO_REF,
    "output_root": str(OUTPUT_ROOT), "dataset_label": DATASET_LABEL,
    "use_preprocessed_data": USE_PREPROCESSED_DATA,
    "raw_dataset_source": RAW_DATASET_SOURCE, "local_dataset_dir": LOCAL_DATASET_DIR,
    "preprocessed_source": PREPROCESSED_SOURCE,
    "save_preprocessed_backup": SAVE_PREPROCESSED_BACKUP,
    "preprocessed_backup_path": str(DRIVE_DATA_BACKUP),
    "data_name": DATA_NAME, "captions_per_image": CAPTIONS_PER_IMAGE,
    "min_word_freq": MIN_WORD_FREQ, "max_len": MAX_LEN,
    "resume_checkpoint": RESUME_CHECKPOINT, "total_epochs": TOTAL_EPOCHS,
    "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS,
    "stop_criteria": STOP_CRITERIA, "print_freq": PRINT_FREQ,
    "encoder_lr": ENCODER_LR, "decoder_lr": DECODER_LR,
    "eval_checkpoint": EVAL_CHECKPOINT, "eval_beam_size": EVAL_BEAM_SIZE,
    "example_image_a": EXAMPLE_IMAGE_A, "example_image_b": EXAMPLE_IMAGE_B,
}
(RUN_DIR / "notebook_config.json").write_text(json.dumps(notebook_config, indent=2, default=str))

environment = {
    "run_id": RUN_ID,
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "torchvision": torchvision.__version__,
    "gdown": version("gdown"),
    "pytorch_cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "repo_url": subprocess.check_output(["git", "remote", "get-url", "origin"], text=True).strip(),
}
(RUN_DIR / "environment.json").write_text(json.dumps(environment, indent=2))
print(json.dumps(environment, indent=2))
print(f"Run folder: {RUN_DIR}")
print("Data mode:", "prepared HDF5/JSON archive" if USE_PREPROCESSED_DATA else "build HDF5/JSON from raw images and captions")
print("Selected source:", PREPROCESSED_SOURCE if USE_PREPROCESSED_DATA else (LOCAL_DATASET_DIR or RAW_DATASET_SOURCE))

def fetch_archive(source, destination):
    source = str(source)
    destination = Path(destination)
    if Path(source).is_file():
        shutil.copy2(source, destination)
    elif source.startswith(("https://", "http://")):
        help_text = subprocess.check_output([sys.executable, "-m", "gdown", "--help"], text=True)
        command = [sys.executable, "-u", "-m", "gdown"]
        if "--fuzzy" in help_text:
            command.append("--fuzzy")
        run_logged(command + [source, "-O", str(destination)], RUN_DIR / "data_download.log")
    else:
        raise FileNotFoundError(f"Expected an existing ZIP path or download URL: {source}")
    return destination


### Checkpoint storage

Authorize Drive API access. Large checkpoints stay on runtime storage and are uploaded with verified replacement, avoiding deleted checkpoint accumulation in Drive trash. Only previous checkpoint IDs in this run folder are managed; no general trash cleanup is performed. See [Drive files.delete](https://developers.google.com/workspace/drive/api/reference/rest/v3/files/delete).


In [ ]:
from google.colab import auth
from drive_checkpoints import DriveCheckpoints, drive_service

auth.authenticate_user()
service = drive_service()

def resolve_drive_folder(path, create=False):
    relative = Path(path).relative_to(Path("/content/drive/MyDrive"))
    folder_id = "root"
    for part in relative.parts:
        escaped = part.replace("\\", "\\\\").replace("'", "\\'")
        matches = service.files().list(
            q=f"'{folder_id}' in parents and name = '{escaped}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false",
            fields="files(id)", pageSize=100,
        ).execute().get("files", [])
        if len(matches) > 1:
            raise RuntimeError(f"Multiple Drive folders named {part}; use a unique path.")
        if matches:
            folder_id = matches[0]["id"]
        elif create:
            folder_id = service.files().create(
                body={"name": part, "mimeType": "application/vnd.google-apps.folder", "parents": [folder_id]},
                fields="id",
            ).execute()["id"]
        else:
            raise FileNotFoundError(path)
    return folder_id

DRIVE_CHECKPOINT_DIR = RUN_DIR / "models_checkpoint"
DRIVE_CHECKPOINT_FOLDER_ID = resolve_drive_folder(DRIVE_CHECKPOINT_DIR, create=True)
print(f"Local checkpoints: {CHECKPOINT_DIR}")
print(f"Drive checkpoint folder ID: {DRIVE_CHECKPOINT_FOLDER_ID}")


## 5. Load images and captions, if needed

This step is skipped when `USE_PREPROCESSED_DATA=True`. Otherwise use the configured raw ZIP source or existing `LOCAL_DATASET_DIR`. Your own processed images can be used here: the ZIP must retain the caption file and `images/` layout expected by RSICC. A single enclosing dataset folder is supported.

For gdown versions that support it, the downloader adds `--fuzzy`; newer versions parse share URLs automatically.


In [ ]:
def dataset_ready():
    return ((DATASET_DIR / "LevirCCcaptions.json").is_file()
            and (DATASET_DIR / "images").is_dir())

if USE_PREPROCESSED_DATA:
    print("Using prepared HDF5/JSON data; raw dataset download is skipped.")
else:
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if not dataset_ready():
        if LOCAL_DATASET_DIR is not None:
            raise FileNotFoundError("LOCAL_DATASET_DIR must contain LevirCCcaptions.json and images/.")
        archive = fetch_archive(RAW_DATASET_SOURCE, Path("/content/dataset_download.zip"))
        print("Extracting raw dataset...", flush=True)
        with zipfile.ZipFile(archive) as zipped:
            zipped.extractall(DATASET_DIR)
        if not dataset_ready():
            candidates = [p.parent for p in DATASET_DIR.rglob("LevirCCcaptions.json")
                          if (p.parent / "images").is_dir()]
            if len(candidates) == 1:
                DATASET_DIR = candidates[0]
        if not dataset_ready():
            raise FileNotFoundError("Expected LevirCCcaptions.json and images/ in the dataset ZIP.")
        archive.unlink()
    print(f"Images and captions: {DATASET_DIR}")


## 6. Prepare training files

`USE_PREPROCESSED_DATA=True` loads exactly the archive selected by `PREPROCESSED_SOURCE` (a Drive ZIP path or URL). It must contain the WORDMAP and TRAIN/VAL/TEST HDF5, CAPTIONS, and CAPLENS files matching `DATA_NAME`. Flat ZIPs and nested data folders are supported. This mode skips image preprocessing entirely.

With `USE_PREPROCESSED_DATA=False`, the selected images and captions are processed with the configured vocabulary settings. No old backup is restored automatically. `SAVE_PREPROCESSED_BACKUP` controls whether the resulting files are zipped to Drive; the default is off. Use a distinct dataset label for each image or caption variant. Resume requires the vocabulary and data expected by the source checkpoint.


In [ ]:
required_files = required_data_files(DATA_DIR, DATA_NAME)
if all(p.is_file() for p in required_files):
    print(f"Training files already available for this run: {DATA_DIR}")
else:
    existing_outputs = [p.name for p in required_files if p.exists()]
    if existing_outputs:
        raise RuntimeError("Partial outputs in this run: " + ", ".join(existing_outputs)
                           + ". Move these generated files before retrying.")
    if USE_PREPROCESSED_DATA:
        if not PREPROCESSED_SOURCE:
            raise ValueError("Set PREPROCESSED_SOURCE to the prepared HDF5/JSON ZIP path or URL.")
        archive = fetch_archive(PREPROCESSED_SOURCE, Path("/content/preprocessed_download.zip"))
        print(f"Restoring the selected prepared data: {PREPROCESSED_SOURCE}", flush=True)
        restore_preprocessed_data(archive, DATA_DIR, DATA_NAME)
        archive.unlink()
    else:
        run_logged([
            sys.executable, "-u", "create_input_files.py",
            "--captions", str(DATASET_DIR / "LevirCCcaptions.json"),
            "--images", str(DATASET_DIR / "images"), "--output_folder", str(DATA_DIR),
            "--captions_per_image", str(CAPTIONS_PER_IMAGE),
            "--min_word_freq", str(MIN_WORD_FREQ), "--max_len", str(MAX_LEN),
        ], RUN_DIR / "preprocessing.log")
    missing = [p.name for p in required_files if not p.is_file()]
    if missing:
        raise RuntimeError("Missing training files: " + ", ".join(missing))

if SAVE_PREPROCESSED_BACKUP and not USE_PREPROCESSED_DATA:
    if DRIVE_DATA_BACKUP.exists():
        raise FileExistsError(f"Backup already exists; choose a new path or disable backup: {DRIVE_DATA_BACKUP}")
    print(f"Saving optional data backup: {DRIVE_DATA_BACKUP}", flush=True)
    DRIVE_DATA_BACKUP.parent.mkdir(parents=True, exist_ok=True)
    archive = shutil.make_archive(str(DATA_DIR.parent / "data_preprocessed"), "zip", DATA_DIR.parent, DATA_DIR.name)
    shutil.copy2(archive, DRIVE_DATA_BACKUP)
    Path(archive).unlink()
else:
    print("No preprocessed-data backup will be written to Drive.")
print(f"Ready: {len(required_files)} training files in {DATA_DIR}")


## 7. Train or resume training

All parameters are set in the configuration cell. The defaults use 20 epochs, batch size 32, and 4 data-loading workers. `PRINT_FREQ` controls the batch reporting interval. Output is streamed live and saved to `train.log`; epoch validation scores are also saved to `training_metrics.jsonl`. Validation runs after every epoch; the test set is evaluated separately below. Reduce the batch size if GPU memory is insufficient. These values do not change the model architecture.

For a new training run, leave `RESUME_CHECKPOINT = None`. To continue, provide the full Drive path to an earlier checkpoint. The regular `checkpoint_...pth.tar` stores the latest completed epoch; `BEST_checkpoint_...pth.tar` stores the best validation BLEU-4 checkpoint. Older runs may only have the best checkpoint, so resuming those runs starts after that checkpoint's epoch.

`TOTAL_EPOCHS` is the total target, not the number of additional epochs. For example, after 20 completed epochs, set it to 30 for up to 10 more. Model weights, optimizer states, best BLEU-4, and the early-stopping counter are restored. New checkpoints also restore scheduler state. Legacy checkpoints lack scheduler state; counters restart (the existing StepLR uses gamma=1). Random-number and data-loader states are not restored, so this is not a bit-for-bit continuation.

Keep the same dataset, vocabulary, and model settings. Restored optimizer learning rates take precedence over the configured initial learning rates. Early stopping remains active: to continue a run that already exhausted patience, set `STOP_CRITERIA` above its saved counter. A resumed run writes into the new run folder and leaves its source checkpoint intact. Checkpoints are saved locally, then uploaded through the Drive API. Only the latest and best checkpoint files are retained in each run folder. The previous file of the same name is permanently deleted only after verifying the new upload size and MD5 checksum. Upload or deletion errors stop training. Keep space for old and new copies during upload. Existing Drive trash is not removed. Do not run concurrent training processes against the same checkpoint folder. If no new best score is reached, select the old best checkpoint explicitly for evaluation.


In [ ]:
command = [
    sys.executable, "-u", "train.py",
    "--data_folder", str(DATA_DIR), "--data_name", DATA_NAME,
    "--savepath", str(CHECKPOINT_DIR),
    "--drive_checkpoint_folder_id", DRIVE_CHECKPOINT_FOLDER_ID,
    "--epochs", str(TOTAL_EPOCHS), "--batch_size", str(BATCH_SIZE),
    "--workers", str(NUM_WORKERS), "--stop_criteria", str(STOP_CRITERIA),
    "--print_freq", str(PRINT_FREQ),
    "--encoder_lr", str(ENCODER_LR), "--decoder_lr", str(DECODER_LR),
]
if RESUME_CHECKPOINT is not None:
    if not Path(RESUME_CHECKPOINT).is_file():
        raise FileNotFoundError(RESUME_CHECKPOINT)
    source = Path(RESUME_CHECKPOINT)
    if source.is_relative_to(Path("/content/drive/MyDrive")):
        source_folder_id = resolve_drive_folder(source.parent, create=False)
        restored = CHECKPOINT_DIR / "resume_source" / source.name
        DriveCheckpoints(source_folder_id, service).download(source.name, restored)
        source = restored
    command += ["--checkpoint", str(source)]

config = {
    "dataset_label": DATASET_LABEL, "data_name": DATA_NAME,
    "use_preprocessed_data": USE_PREPROCESSED_DATA,
    "data_source": PREPROCESSED_SOURCE if USE_PREPROCESSED_DATA else (LOCAL_DATASET_DIR or RAW_DATASET_SOURCE),
    "preprocessed_backup": str(DRIVE_DATA_BACKUP) if SAVE_PREPROCESSED_BACKUP else None,
    "resume_checkpoint": RESUME_CHECKPOINT,
    "drive_checkpoint_folder_id": DRIVE_CHECKPOINT_FOLDER_ID,
    "total_epochs": TOTAL_EPOCHS, "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS, "stop_criteria": STOP_CRITERIA, "print_freq": PRINT_FREQ,
    "encoder_lr": ENCODER_LR, "decoder_lr": DECODER_LR,
    "command": command,
}
(RUN_DIR / "training_config.json").write_text(json.dumps(config, indent=2, default=str))
try:
    run_logged(command, RUN_DIR / "train.log")
finally:
    metrics_path = CHECKPOINT_DIR / "training_metrics.jsonl"
    if metrics_path.is_file():
        shutil.copy2(metrics_path, RUN_DIR / metrics_path.name)


## 8. Evaluate one checkpoint on the test set

Choose a single file with `EVAL_CHECKPOINT`. By default, this selects the best checkpoint from the current run. You can select the latest checkpoint or a checkpoint from an earlier run instead.

The scripts read every file in their checkpoint directory. This cell creates a temporary directory containing only the selected checkpoint. Each evaluation gets its own results folder, including the selected checkpoint path and command.

In [ ]:
import tempfile

CHECKPOINT_NAME = "checkpoint_resnet101_MCCFormers_diff_as_Q_trans.pth.tar"
EVAL_CHECKPOINT = Path(EVAL_CHECKPOINT) if EVAL_CHECKPOINT is not None else CHECKPOINT_DIR / ("BEST_" + CHECKPOINT_NAME)
if not Path(EVAL_CHECKPOINT).is_file():
    raise FileNotFoundError(f"Select an existing checkpoint: {EVAL_CHECKPOINT}")

EVAL_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
DRIVE_RESULTS_DIR = RUN_DIR / "eval_results" / EVAL_ID
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=False)
with tempfile.TemporaryDirectory(prefix="rsicc_eval_", dir="/content") as staging:
    os.symlink(Path(EVAL_CHECKPOINT).resolve(), Path(staging) / Path(EVAL_CHECKPOINT).name)
    command = [sys.executable, "-u", "eval.py", "--data_folder", str(DATA_DIR),
               "--data_name", DATA_NAME, "--path", staging, "--Split", "TEST", "--beam_size", str(EVAL_BEAM_SIZE)]
    (DRIVE_RESULTS_DIR / "evaluation_config.json").write_text(json.dumps(
        {"checkpoint": str(EVAL_CHECKPOINT), "command": command}, indent=2))
    run_logged(command, DRIVE_RESULTS_DIR / "eval.log")
shutil.copytree(PROJECT_DIR / "eval_results_fortest" / "TEST", DRIVE_RESULTS_DIR / "TEST")
for score_file in (DRIVE_RESULTS_DIR / "TEST").glob("*_scores.json"):
    print(json.dumps(json.loads(score_file.read_text()), indent=2))
print(f"Evaluation results: {DRIVE_RESULTS_DIR}")


## 9. Generate a caption for an example image pair

This uses the single checkpoint selected above. The generated caption is also copied to this run's Drive folder.

In [ ]:
if not Path(EVAL_CHECKPOINT).is_file():
    raise FileNotFoundError(EVAL_CHECKPOINT)
with tempfile.TemporaryDirectory(prefix="rsicc_caption_", dir="/content") as staging:
    os.symlink(Path(EVAL_CHECKPOINT).resolve(), Path(staging) / Path(EVAL_CHECKPOINT).name)
    run_logged([sys.executable, "-u", "caption.py",
                    "--img_A", str(EXAMPLE_IMAGE_A),
                    "--img_B", str(EXAMPLE_IMAGE_B),
                    "--path", staging, "--data_folder", str(DATA_DIR),
                    "--data_name", DATA_NAME], RUN_DIR / "caption.log")
CAPTION_DIR = RUN_DIR / "example_captions" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8])
CAPTION_DIR.mkdir(parents=True, exist_ok=False)
(CAPTION_DIR / "caption_config.json").write_text(json.dumps(
    {"checkpoint": str(EVAL_CHECKPOINT), "img_A": str(EXAMPLE_IMAGE_A),
     "img_B": str(EXAMPLE_IMAGE_B)}, indent=2))
for path in [PROJECT_DIR / "eval_results" / "resnet101_MCCFormers_diff_as_Q_trans_res.json"]:
    shutil.copy2(path, CAPTION_DIR)
    print(path.read_text())
print(f"Example captions: {CAPTION_DIR}")
